<a href="https://colab.research.google.com/github/dtsri/partymetrics/blob/main/event_analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Birthday / Event Planning Analytics - Synthetic Data Generator
-----------------------------------------------------------------
Generates 4 related CSV files for a SQL + Power BI + Python portfolio project:
  1. customers.csv
  2. events.csv
  3. vendor_orders.csv
  4. cakes.csv

Designed to be run in Google Colab. Business rules (correlations) are
deliberately baked in so that your analysis "discovers" real patterns
instead of finding random noise. All data is 100% synthetic.

Usage in Colab:
  1. Paste this whole script into a cell and run it.
  2. It will create the 4 CSVs in /content/ and auto-download them.
"""

import numpy as np
import pandas as pd
import random
from datetime import datetime, timedelta

# Try to use Faker if available; otherwise use a manual name list fallback.
try:
    from faker import Faker
    fake = Faker()
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "faker", "-q"])
    from faker import Faker
    fake = Faker()

# ---------------------------------------------------------------
# CONFIG - tweak these to control dataset size
# ---------------------------------------------------------------
NUM_CUSTOMERS = 300
NUM_EVENTS = 1200
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
Faker.seed(RANDOM_SEED)

CITIES = ["Visakhapatnam", "Hyderabad", "Bangalore", "Chennai", "Vijayawada", "Pune", "Mumbai", "Delhi"]
SEGMENTS = ["Family", "Corporate", "Friend Group"]
THEMES = ["Superhero", "Princess", "Jungle Safari", "Minimalist", "Cartoon Characters",
          "Sports", "Space", "Unicorn", "Vintage", "Beach Party"]
AGE_GROUPS = ["Kids (1-5)", "Kids (6-12)", "Teen (13-19)", "Adult (20+)"]
VENUE_TYPES = ["Indoor", "Outdoor", "Home"]
WEATHER = ["Sunny", "Rainy", "Cloudy", "Stormy"]
VENDOR_CATEGORIES = ["Cake", "Decoration", "Catering", "Balloon", "Return Gifts", "Photography"]
CAKE_FLAVORS = ["Chocolate", "Vanilla", "Red Velvet", "Butterscotch", "Black Forest", "Fruit", "Pineapple"]
CUSTOMIZATION_LEVELS = ["Basic", "Premium", "Custom"]

# ---------------------------------------------------------------
# 1. CUSTOMERS
# ---------------------------------------------------------------
customers = []
for i in range(1, NUM_CUSTOMERS + 1):
    customers.append({
        "customer_id": i,
        "customer_name": fake.name(),
        "segment": random.choices(SEGMENTS, weights=[0.6, 0.15, 0.25])[0],
        "city": random.choice(CITIES),
        "repeat_customer": random.choices([True, False], weights=[0.3, 0.7])[0],
    })
customers_df = pd.DataFrame(customers)

# ---------------------------------------------------------------
# 2. EVENTS  (with intentional business-rule correlations)
# ---------------------------------------------------------------
start_date = datetime(2024, 1, 1)
events = []

for event_id in range(1, NUM_EVENTS + 1):
    customer = customers_df.sample(1).iloc[0]
    event_date = start_date + timedelta(days=random.randint(0, 730))
    month = event_date.month
    day_of_week = event_date.strftime("%A")
    is_weekend = day_of_week in ["Saturday", "Sunday"]

    # Seasonality: more events + higher weather risk in Apr-Jun (summer) and Oct-Dec (festive)
    season = "Summer" if month in [4, 5, 6] else "Monsoon" if month in [7, 8, 9] else \
             "Festive" if month in [10, 11, 12] else "Winter"

    venue_type = random.choices(VENUE_TYPES, weights=[0.4, 0.35, 0.25])[0]

    # Weather more likely bad in Monsoon, especially affects Outdoor venues
    if season == "Monsoon":
        weather = random.choices(WEATHER, weights=[0.2, 0.5, 0.2, 0.1])[0]
    else:
        weather = random.choices(WEATHER, weights=[0.55, 0.15, 0.25, 0.05])[0]

    guest_count = int(np.clip(np.random.normal(80, 40), 10, 300))
    age_group = random.choice(AGE_GROUPS)
    theme = random.choice(THEMES)

    planning_days = int(np.clip(np.random.normal(25, 10), 3, 90))

    budget_planned = int(np.clip(np.random.normal(40000, 15000), 8000, 150000))
    # scale budget up a bit with guest count
    budget_planned = int(budget_planned + guest_count * 150)

    # ---- KEY BUSINESS RULE ----
    # Outdoor events with >120 guests overshoot budget by ~18% on average
    overrun_pct = 0.0
    if venue_type == "Outdoor" and guest_count > 120:
        overrun_pct = np.random.normal(0.18, 0.05)
    else:
        overrun_pct = np.random.normal(0.04, 0.05)
    overrun_pct = max(overrun_pct, -0.1)
    budget_actual = int(budget_planned * (1 + overrun_pct))

    # Cancellations more likely: Outdoor + Stormy/Rainy weather
    cancel_prob = 0.02
    if venue_type == "Outdoor" and weather in ["Rainy", "Stormy"]:
        cancel_prob = 0.22
    cancelled = random.random() < cancel_prob

    # Satisfaction: lower when budget overrun is high or event cancelled
    base_satisfaction = np.random.normal(4.2, 0.6)
    if overrun_pct > 0.15:
        base_satisfaction -= 0.8
    if cancelled:
        base_satisfaction -= 2.0
    satisfaction_score = int(np.clip(round(base_satisfaction), 1, 5))

    events.append({
        "event_id": event_id,
        "customer_id": int(customer["customer_id"]),
        "event_date": event_date.strftime("%Y-%m-%d"),
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "season": season,
        "city": customer["city"],
        "theme": theme,
        "age_group": age_group,
        "guest_count": guest_count,
        "venue_type": venue_type,
        "weather_condition": weather,
        "budget_planned": budget_planned,
        "budget_actual": budget_actual,
        "budget_overrun_pct": round(overrun_pct * 100, 1),
        "planning_days": planning_days,
        "satisfaction_score": satisfaction_score,
        "cancelled": cancelled,
    })

events_df = pd.DataFrame(events)

# ---------------------------------------------------------------
# 3. VENDOR ORDERS (one or more per event)
# ---------------------------------------------------------------
# Give each vendor category a different baseline delay/reliability profile
vendor_delay_profile = {
    "Cake": (10, 5),          # (mean delay minutes, std) when late
    "Decoration": (30, 15),
    "Catering": (20, 10),
    "Balloon": (15, 8),
    "Return Gifts": (12, 6),
    "Photography": (5, 3),
}
vendor_late_prob = {
    "Cake": 0.10,
    "Decoration": 0.28,   # decoration vendors are least reliable
    "Catering": 0.18,
    "Balloon": 0.15,
    "Return Gifts": 0.08,
    "Photography": 0.06,
}

vendor_orders = []
vendor_order_id = 1
for _, ev in events_df.iterrows():
    # Each event uses 3-6 vendor categories
    categories_used = random.sample(VENDOR_CATEGORIES, k=random.randint(3, 6))
    for cat in categories_used:
        is_late = random.random() < vendor_late_prob[cat]
        delay_mean, delay_std = vendor_delay_profile[cat]
        delay_minutes = int(np.clip(np.random.normal(delay_mean, delay_std), 0, 180)) if is_late else 0
        status = "Cancelled" if ev["cancelled"] and random.random() < 0.3 else \
                 "Late" if is_late else "Delivered"

        base_cost = {
            "Cake": 2500, "Decoration": 8000, "Catering": 15000,
            "Balloon": 3000, "Return Gifts": 4000, "Photography": 6000
        }[cat]
        cost = int(np.clip(np.random.normal(base_cost, base_cost * 0.3), base_cost * 0.4, base_cost * 2))

        vendor_orders.append({
            "vendor_order_id": vendor_order_id,
            "event_id": int(ev["event_id"]),
            "vendor_category": cat,
            "vendor_name": f"{cat} Vendor {random.randint(1, 25)}",
            "cost": cost,
            "delay_minutes": delay_minutes,
            "status": status,
        })
        vendor_order_id += 1

vendor_orders_df = pd.DataFrame(vendor_orders)

# ---------------------------------------------------------------
# 4. CAKES (sub-detail, one per event)
# ---------------------------------------------------------------
cakes = []
for _, ev in events_df.iterrows():
    customization = random.choices(CUSTOMIZATION_LEVELS, weights=[0.4, 0.35, 0.25])[0]
    base_cost = {"Basic": 1200, "Premium": 2800, "Custom": 4500}[customization]
    size_kg = round(np.clip(np.random.normal(1.5, 0.7), 0.5, 5), 1)
    cost = int(base_cost * size_kg / 1.5)

    cakes.append({
        "cake_id": int(ev["event_id"]),  # 1:1 with event for simplicity
        "event_id": int(ev["event_id"]),
        "flavor": random.choice(CAKE_FLAVORS),
        "size_kg": size_kg,
        "customization_level": customization,
        "cost": cost,
    })

cakes_df = pd.DataFrame(cakes)

# ---------------------------------------------------------------
# SAVE + DOWNLOAD (Colab)
# ---------------------------------------------------------------
customers_df.to_csv("customers.csv", index=False)
events_df.to_csv("events.csv", index=False)
vendor_orders_df.to_csv("vendor_orders.csv", index=False)
cakes_df.to_csv("cakes.csv", index=False)

print("Generated:")
print(f"  customers.csv      -> {len(customers_df)} rows")
print(f"  events.csv         -> {len(events_df)} rows")
print(f"  vendor_orders.csv  -> {len(vendor_orders_df)} rows")
print(f"  cakes.csv          -> {len(cakes_df)} rows")

# Auto-download in Colab (safe to ignore error if run outside Colab)
try:
    from google.colab import files
    for fname in ["customers.csv", "events.csv", "vendor_orders.csv", "cakes.csv"]:
        files.download(fname)
except ImportError:
    print("\n(Not running in Colab - files saved locally instead of auto-downloading.)")

Generated:
  customers.csv      -> 300 rows
  events.csv         -> 1200 rows
  vendor_orders.csv  -> 5393 rows
  cakes.csv          -> 1200 rows


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>